In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [3]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/training"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/dataset-test")

Found 187 branch digraphs...


Computing OD/Macula centers:   0%|          | 0/187 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")

Processing...
Done!


In [ ]:
len(dataset)

## Graph Augment


In [5]:
ID = 0

In [ ]:
m, digraph, _ = dataset.jppype_show(ID, augment=True)
m

In [6]:
dataset.get(20, augment=True)
%timeit dataset.get(20, augment=False)

241 ms ± 16.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%timeit dataset.get(20)
dataset.cfg.augment.elastic = False
%timeit dataset.get(20)

In [ ]:
import numpy as np

np.argmax([False, False, False, False, False])

In [7]:
from fundus_toolkits.transform import ElasticTransform, IdentityTransform
import numpy as np

sample = dataset.get_sample(0)

fundus_img = sample.fundus.image
graph = next(iter(sample.graphes.values()))
fundus_img = fundus_img.transpose(1, 2, 0).astype(np.float32)  # C,H,W -> H,W,C
shape = fundus_img.shape[0], fundus_img.shape[1]

elastic = ElasticTransform.random(shape, displacement_std=120, smoothing_size=200)
identity = IdentityTransform()


In [ ]:
graph.geometric_data().clear_attribute(all_except="CALIBRE")
graph.transform(elastic, inplace=False)
%timeit graph.transform(elastic, inplace=False)
%timeit elastic.warp(fundus_img, warped_domain="same")

69.2 ms ± 3.16 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
36.2 ms ± 427 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
curves = np.concatenate(graph.geometric_data().branch_curve(), axis=0)

In [23]:
curve = graph.geometric_data().branch_curve(0)
curve.shape

(66, 2)

In [21]:
%timeit np.concatenate(graph.geometric_data().branch_curve(), axis=0)

40.2 μs ± 403 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [24]:
from fundus_vessels_toolkit.utils.cpp_extensions.fvt_cpp import (
    inverse_displacement,
    vec_bilinear_interpolate,
)


def transform(curve):
    disp_t = torch.from_numpy(elastic.displacement.astype(np.float32))
    src_t = torch.from_numpy(curve.astype(np.float32)).reshape(-1, 2)
    inv_d = inverse_displacement(disp_t, src_t, 50, 0.5).numpy()
    return curve + inv_d.reshape(curve.shape)

In [ ]:
%timeit elastic.transform(curve)
%timeit transform(curve)

disp_t = torch.from_numpy(elastic.displacement.astype(np.float32))
src_t = torch.from_numpy(curve.astype(np.float32)).reshape(-1, 2)
%timeit torch.from_numpy(elastic.displacement.astype(np.float32))
%timeit torch.from_numpy(curve.astype(np.float32)).reshape(-1, 2)
%timeit inverse_displacement(disp_t, src_t, 50, 0.5)

265 μs ± 19.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
253 μs ± 4.05 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
8.81 μs ± 72.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
%timeit elastic.transform(curves)

2.4 ms ± 34.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
graph.geometric_data()._branch_data_dict.keys()

In [ ]:
len(digraph.graph.geometric_data().branch_curve())

In [ ]:
curve = digraph.graph.branch(0).curve()
elastic = ElasticTransform.random(shape, displacement_std=120, smoothing_size=200)
%timeit elastic.transform(curve)

In [ ]:
m.views[0].goto(dataset.get_digraph(0).graph.branch(24).midpoint().xy, 3)

In [ ]:
for i, data in enumerate(DataLoader(dataset, batch_size=8, num_workers=4)):
    ...